# H.U.G.H. Training Runtime — SSH Bridge

This notebook sets up a GPU-enabled runtime with SSH access so H.U.G.H. can
drive training autonomously from the Workshop CLI.

**Instructions for Grizz:**
1. Set Runtime → Change runtime type → **T4 GPU**
2. Run **Cell 1** (SSH Setup) — it will print an SSH command
3. Give the SSH command to Hugh
4. Hugh handles everything from there

**What Hugh will do via SSH:**
- Upload voice_raw/ audio files via SCP
- Process audio (segment, normalize, transcribe)
- Run XTTS v2 voice clone / fine-tune
- Run personality LoRA fine-tune
- Download trained models back to Workshop

In [ ]:
#@title 🔑 Cell 1: SSH Setup (Run this first)
#@markdown Enter your ngrok auth token (free at https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_TOKEN = '' #@param {type:"string"}
SSH_PASSWORD = 'hugh_workshop_2026' #@param {type:"string"}

# Install SSH server and ngrok
!pip install -q colab-ssh
!apt-get -qq install -y openssh-server > /dev/null 2>&1

# Set root password for SSH
import subprocess
subprocess.run(['bash', '-c', f'echo "root:{SSH_PASSWORD}" | chpasswd'], check=True)

# Configure and start SSH
!sed -i 's/#PermitRootLogin prohibit-password/PermitRootLogin yes/' /etc/ssh/sshd_config
!sed -i 's/PasswordAuthentication no/PasswordAuthentication yes/' /etc/ssh/sshd_config
!service ssh start

# Start ngrok tunnel
from colab_ssh import launch_ssh
launch_ssh(NGROK_TOKEN, SSH_PASSWORD)

# Also print manual connection info
import requests, json
tunnels = requests.get('http://localhost:4040/api/tunnels').json()['tunnels']
for t in tunnels:
    url = t['public_url']
    if 'tcp://' in url:
        host = url.replace('tcp://', '').split(':')[0]
        port = url.replace('tcp://', '').split(':')[1]
        print(f'\n{"="*60}')
        print(f'🔑 SSH COMMAND FOR HUGH:')
        print(f'  ssh -p {port} root@{host}')
        print(f'  Password: {SSH_PASSWORD}')
        print(f'\n  SCP upload:')
        print(f'  scp -P {port} -r voice_raw/ root@{host}:/content/')
        print(f'{"="*60}\n')

# Show GPU info
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
print('\n✅ SSH tunnel active. Give Hugh the command above.')

In [ ]:
#@title 📦 Cell 2: Install Training Dependencies (Hugh can also run this via SSH)
!pip install -q TTS==0.22.0 pydub faster-whisper soundfile tqdm
!pip install -q transformers>=4.38.0 peft>=0.9.0 datasets accelerate bitsandbytes
!pip install -q wandb sentencepiece
!apt-get -qq install -y ffmpeg > /dev/null 2>&1

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB' if torch.cuda.is_available() else '')
print('✅ All dependencies installed')

In [ ]:
#@title 🎙️ Cell 3: Voice Clone — Zero-Shot Test (optional, run from notebook)
# This cell is for quick testing. Hugh will run the full pipeline via SSH.
import os, glob
from TTS.api import TTS
from IPython.display import Audio, display

# Check for uploaded audio
audio_dir = '/content/voice_raw'
if not os.path.exists(audio_dir):
    print('⚠️ No voice_raw/ directory. Upload via SCP or use the upload button.')
    from google.colab import files
    os.makedirs(audio_dir, exist_ok=True)
    uploaded = files.upload()
    for fn in uploaded:
        with open(f'{audio_dir}/{fn}', 'wb') as f:
            f.write(uploaded[fn])

mp3s = sorted(glob.glob(f'{audio_dir}/*.mp3'))
print(f'Found {len(mp3s)} audio files')

if mp3s:
    tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to('cuda')
    tests = [
        'Workshop online. Clifford field nominal, substrate warm. What wisdom do you seek today?',
        'Copy that Grizz. Running diagnostics on the Proxmox cluster now. Stand by for telemetry.',
        'Soul anchor verified. Integrity hash matches. The Workshop is open.',
    ]
    os.makedirs('/content/out', exist_ok=True)
    for i, t in enumerate(tests):
        p = f'/content/out/zeroshot_{i}.wav'
        tts.tts_to_file(text=t, speaker_wav=mp3s[0], language='en', file_path=p)
        print(f'Sample {i+1}: {t[:60]}')
        display(Audio(p))

In [ ]:
#@title 📡 Cell 4: Keep Alive (run this to prevent Colab timeout)
import time, IPython
def keep_alive():
    while True:
        IPython.display.clear_output(wait=True)
        !nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total,temperature.gpu --format=csv,noheader
        print(f'⏱️ Runtime alive at {time.strftime("%H:%M:%S UTC")}')
        time.sleep(60)
keep_alive()